# 1. **Fundamentals & Scaling**
* **Vertical Scaling:** Increasing the power (CPU, RAM) of a single machine.
* **Horizontal Scaling:** Adding more machines to your pool (the industry standard for HLD).
* **Serverless:** Functions-as-a-Service (e.g., AWS Lambda). You care about logic; the provider scales the infra.
* **The Internet:** Understanding DNS, IP, and how a request travels from a browser to a server.

### 🔹 Vertical Scaling
**What interviewers want to hear:**
- You understand it’s a **simplicity-first** strategy with a hard ceiling.
- You know when it’s the *right* choice and how to plan the escape hatch.

| Senior Lens | Details |
|-------------|---------|
| **Trade-offs** | Fast to implement, no distributed complexity. But hits hardware limits, creates a SPOF, requires downtime for upgrades, and has non-linear cost at high tiers. |
| **When it makes sense** | Early-stage products, stateful workloads (e.g., relational DBs with complex joins), latency-sensitive single-threaded apps, temporary spikes while horizontal migration is planned. |
| **When to avoid** | Predictable high traffic, multi-region requirements, strict HA/SLA needs, teams ready for distributed patterns. |
| **Interview Q** | *"When would you choose vertical scaling over horizontal?"* |
| **Senior Answer Framework** | `"I’d pick vertical when distributed complexity outweighs scaling benefits: e.g., a monolithic legacy app, a relational DB with heavy ACID joins, or during rapid MVP iteration. I’d monitor CPU/RAM saturation, set alerts at ~70%, and pre-plan a horizontal migration path (stateless extraction, read replicas, sharding) before hitting vendor max instances. Vertical is a tactical scaling lever, not a long-term strategy."` |

---

### 🔹 Horizontal Scaling
**What interviewers want to hear:**
- You know scaling out **shifts complexity to coordination**, not magic.
- You can design for statelessness, partitioning, failure domains, and graceful degradation.

| Senior Lens | Details |
|-------------|---------|
| **Trade-offs** | Near-linear elasticity & fault tolerance. But requires stateless design, load balancing, service discovery, distributed state management, and introduces network latency & partition risks. |
| **Core prerequisites** | Stateless compute, shared-nothing architecture, consistent hashing/sharding, idempotent operations, health checks & autoscaling policies. |
| **When it breaks first** | State synchronization, distributed transactions, cross-AZ network partitions, cascading failures from misconfigured autoscaling, database connection pool exhaustion. |
| **Interview Q** | *"How do you scale a stateful service horizontally?"* |
| **Senior Answer Framework** | `"True horizontal scaling requires stateless frontends. For stateful components, I’d partition data (sharding/consistent hashing), route requests deterministically, and accept eventual consistency where possible. I’d use a distributed cache for hot keys, implement idempotent retries, and design for partial failure (circuit breakers, fallback responses). If strong consistency is required, I’d limit horizontal scope to read replicas and keep writes on a controlled primary cluster, explicitly trading latency for correctness."` |

---

### 🔹 Serverless (FaaS)
**What interviewers want to hear:**
- You understand it’s **not free or infinitely scalable**, and it shifts cost/complexity curves.
- You know its sweet spots, failure modes, and how to architect around limits.

| Senior Lens | Details |
|-------------|---------|
| **Trade-offs** | Zero infra ops, fine-grained billing, auto-scales to zero. But cold starts, execution timeouts (usually 15m), memory/package limits, vendor lock-in, and observability gaps. Cost inverts at high sustained load. |
| **When it makes sense** | Event-driven workflows, async processing, sporadic/unpredictable traffic, glue code, ML inference batches, edge compute. |
| **When to avoid** | Long-running processes, predictable high-throughput APIs, strict p99 latency SLAs, complex stateful transactions, teams needing deep infra control. |
| **Interview Q** | *"When does serverless become a bad choice?"* |
| **Senior Answer Framework** | `"Serverless shines for bursty, event-driven workloads but struggles with sustained high load due to cost inversion and cold starts. I’d avoid it for p99 < 100ms APIs, long-running jobs, or systems needing deep custom kernel/network tuning. If adopted, I’d mitigate cold starts with provisioned concurrency, design idempotent event handlers, use step functions for orchestration, and instrument distributed tracing early. I also model cost at 2x projected peak traffic before committing."` |

---

### 🔹 The Internet (DNS, IP, Request Flow)
**What interviewers want to hear:**
- You treat the network as **unreliable by default**.
- You understand latency budgets, caching layers, TLS overhead, and how failures cascade.

| Senior Lens | Details |
|-------------|---------|
| **Trade-offs** | Global reach, standard protocols. But DNS caching/TTLs cause stale routing, TLS handshakes add latency, TCP head-of-line blocking exists (mitigated by HTTP/3/QUIC), BGP/DNS outages are real. |
| **Senior concerns** | DNS TTL strategy, geo-routing/anycast, CDN edge vs origin, TLS termination points, connection pooling, retry budgets, rate limiting at the edge, certificate lifecycle management. |
| **When it breaks first** | DNS cache poisoning/propagation delays, cert expiry, cross-region latency spikes, DDoS overwhelming origin, misconfigured CDN caching causing stale data. |
| **Interview Q** | *"How do you reduce global request latency while maintaining data freshness?"* |
| **Senior Answer Framework** | `"I’d push static/semi-static content to a CDN with edge caching, use DNS anycast + low TTLs for failover, terminate TLS at the edge, and leverage HTTP/3 to avoid head-of-line blocking. For dynamic data, I’d use stale-while-revalidate patterns, edge compute for lightweight personalization, and regional origin clusters with read replicas. I’d set explicit SLOs for p95/p99 latency, monitor DNS resolution times, and implement retry budgets with exponential backoff. Freshness vs latency is a product decision; I’d expose cache headers and versioned APIs so clients can opt in."` |

---

### 🧠 Cross-Cutting Senior Interview Themes
Interviewers evaluate you on these dimensions, not definitions:
1. **Trade-off articulation**: Every choice sacrifices something. Name it.
2. **Failure awareness**: What breaks first under load? How do you degrade gracefully?
3. **Cost & ops maturity**: Scaling isn't free. Mention FinOps, observability, runbooks, capacity planning.
4. **Migration paths**: No system is perfect forever. How do you evolve without downtime?
5. **Product alignment**: Technical decisions map to SLAs, user experience, and business constraints.

---

### 📐 How to Structure Senior-Level Answers in Interviews
Use this framework consistently:
1. **Clarify constraints**: Traffic pattern, latency SLA, consistency needs, budget, team maturity.
2. **State assumptions**: "Assuming p99 < 200ms and 10k RPS steady state..."
3. **Present options with trade-offs**: "Vertical gives simplicity but caps at X. Horizontal scales but requires Y. I'd choose Z because..."
4. **Address failure modes**: "Under network partition, we fall back to..."
5. **Validate & iterate**: "I'd load test at 1.5x peak, monitor SLO burn rate, and adjust autoscaling thresholds."

---

### 🗂️ Quick Reference: Senior Decision Matrix
| Concept | Default Choice | Avoid When | Senior Mitigation |
|---------|----------------|------------|-------------------|
| Vertical | Early stage, stateful DBs, latency-critical single node | HA required, predictable high load, multi-region | Monitor saturation, plan stateless extraction, schedule maintenance windows |
| Horizontal | Stateless APIs, microservices, elastic workloads | Complex distributed state, tight consistency, low ops maturity | Consistent hashing, idempotency, circuit breakers, cross-AZ deployment |
| Serverless | Event-driven, sporadic traffic, async glue | Sustained high load, strict p99, long-running jobs | Provisioned concurrency, step functions, distributed tracing, cost modeling |
| Internet/Request Flow | Global user base, low-latency expectations | Ignoring DNS/TLS/CDN layers, assuming reliable network | Anycast DNS, edge caching, HTTP/3, retry budgets, cert automation, SLO tracking |